<a href="https://colab.research.google.com/github/Thanwarin/robot-webots/blob/main/export_representation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This code snippet splits a trained emotion recognition model into two parts for more flexible usage:

# Load the full emotion model

The model is loaded from a .h5 file (EMOTION_MODEL_PATH) and its architecture is summarized.

# Create a representation model

- Extracts the penultimate layer (just before softmax) as feature embeddings (128-d vector).
- Builds a new rep_model that outputs these embeddings instead of predictions.
- Saves it separately (REP_SAVE_PATH).
- Useful for tasks like feature extraction, transfer learning, or clustering.

# Create a classifier-only model

- Takes 128-d embeddings as input and applies the original softmax classifier.
- Builds classifier_model for scenarios where you already have representations and only need the final classification step.
- Saves it separately (CLASSIFIER_SAVE_PATH).

Summary: This setup allows decoupling feature extraction and classification, making the model more modular for experiments or downstream tasks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from tensorflow.keras.models import load_model, Model
import sys, os
from tensorflow.keras import Input
from tensorflow.keras.models import Model

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/ml_models"
model_name = 'ver6'

In [ ]:
import os
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input

# =====================================
# Paths for saving models
# =====================================
SAVE_PATH = "/content/drive/MyDrive/ml_models"
model_name = "ver9_final"

EMOTION_MODEL_PATH = os.path.join(SAVE_PATH, f"emotion_model_{model_name}.h5")
REP_SAVE_PATH = os.path.join(SAVE_PATH, f"representation_model_{model_name}.h5")
CLASSIFIER_SAVE_PATH = os.path.join(SAVE_PATH, f"emotion_classifier_only_{model_name}.h5")

# =====================================
# Load the full emotion recognition model
# =====================================
model = load_model(EMOTION_MODEL_PATH)
model.summary()

# =====================================
# Extract the penultimate layer (representation)
# =====================================
# Usually the second-to-last layer is a Dense layer (e.g., 128-d)
rep_output_layer = model.layers[-2].output

# Create a representation model: input -> 128-d features
rep_model = Model(inputs=model.input, outputs=rep_output_layer)
rep_model.save(REP_SAVE_PATH)
print(f"Saved representation model -> {REP_SAVE_PATH}")

# =====================================
# Create classifier-only model
# =====================================
# Input: 128-d vector (from representation model)
classifier_input = Input(shape=rep_output_layer.shape[1:])

# Reuse the last layer of the original model for classification
classifier_output = model.layers[-1](classifier_input)

# Build classifier-only model
classifier_model = Model(classifier_input, classifier_output)
classifier_model.save(CLASSIFIER_SAVE_PATH)
print(f"Saved classifier-only model -> {CLASSIFIER_SAVE_PATH}")



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 96, 96, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 48, 48,    │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 48, 48,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 48, 48,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 48, 48,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 48, 48,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 48, 48,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 48, 48,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 48, 48,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 48, 48,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 48, 48,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 48, 48,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 49, 49,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 24, 24,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 24, 24,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 24, 24,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 24, 24,    │      2,304 │ block_1_depthwis

 Total params: 2,422,857 (9.24 MB)

 Trainable params: 2,019,975 (7.71 MB)

 Non-trainable params: 402,880 (1.54 MB)

 Optimizer params: 2 (12.00 B)

Saved representation model -> /content/drive/MyDrive/ml_models/representation_model_ver6.h5
Saved classifier-only model -> /content/drive/MyDrive/ml_models/emotion_classifier_only_ver6.h5


In [ ]:
# =====================================
# Summary:
# - Full model: original emotion recognition model
# - Representation model: outputs feature embeddings (128-d)
# - Classifier-only: takes embeddings and outputs softmax predictions
# This allows modular usage for downstream tasks, transfer learning, or feature analysis.
# =====================================
